In [14]:
using Revise
using Pkg

ENV["PYTHON"] = Sys.which("python")
ENV["PYCALL_JL_RUNTIME_PYTHON"] = Sys.which("python")
Pkg.build("PyCall")
using FileIO
using JLD2
include("../src/DistributionallyRobust.jl")
using .DistributionallyRobust

    Building Conda ─→ `~/anaconda3/envs/DRCC-MPC/share/julia/scratchspaces/44cfe95a-1eb2-52ea-b672-e2afdf69b78f/b19db3927f0db4151cb86d073689f2428e524576/build.log`
    Building PyCall → `~/anaconda3/envs/DRCC-MPC/share/julia/scratchspaces/44cfe95a-1eb2-52ea-b672-e2afdf69b78f/9816a3826b0ebf49ab4926e2b18842ad8b5c8f04/build.log`
┌ Info: Number of Julia Thread(s): 1
└ @ Main.DistributionallyRobust /home/kanghyunryu/DRCC-MPC/src/DistributionallyRobust.jl:39
┌ Info: CUDA Device: NVIDIA GeForce RTX 3060
└ @ Main.DistributionallyRobust /home/kanghyunryu/DRCC-MPC/src/DistributionallyRobust.jl:40
┌ Info: Python executable used by PyCall: /home/kanghyunryu/anaconda3/envs/DRCC-MPC/bin/python
└ @ Main.DistributionallyRobust /home/kanghyunryu/DRCC-MPC/src/DistributionallyRobust.jl:41


In [74]:
include("$(@__DIR__)/../scripts/default_params/params_drc_data_trajectron.jl");

epsilon = 0.05;

test_data_name = "hotel_test.pkl";                                                  # test data set name
test_scene_id = 0;                                                                  # test data id
start_time_idx = 401;                                                               # start time index in test data
ego_pos_init_vec = [-1.5, -8.5] .+ [-1.393743, 2.978962];                           # initial ego position [x, y] [m]
ego_pos_goal_vec = [3.5, 0.0]   .+ [-1.393743, 2.978962];                           # goal ego position [x, y] [m]
target_speed = 1.0;                                                                 # target speed [m/s]
sim_horizon = 10.0;       

include("$(@__DIR__)/../scripts/parameter_setup_drc.jl");

In [75]:
scene_loader, controller, w_init, measurement_schedule, target_trajectory, target_speed =
    controller_setup(scene_param,
                    predictor_param,
                    prediction_device=prediction_device,
                    cost_param=cost_param,
                    cnt_param=cnt_param,
                    dtc=dtc,
                    ego_pos_init_vec=ego_pos_init_vec,
                    ego_pos_goal_vec=ego_pos_goal_vec,
                    target_speed=target_speed,
                    sim_horizon=sim_horizon,
                    verbose=true);

Scene Mode: data
Prediction Mode: trajectron
Deterministic Prediction: false
Loaded evaluation data from /home/kanghyunryu/DRCC-MPC/Trajectron-plus-plus/experiments/processed/hotel_test.pkl
Looking at the hotel_test.pkl sequence, data_id 0, start_idx 401
Loaded Trajectron model from /home/kanghyunryu/DRCC-MPC/Trajectron-plus-plus/experiments/pedestrians/models/hotel_attention_radius_3/model_registrar-100.pt


In [76]:
result, ~, ~ = evaluate(scene_loader, controller, w_init, ego_pos_goal_vec,
                  target_speed, measurement_schedule, target_trajectory,
                  pos_error_replan);

┌ Warning: Time 3.60 [sec]: All samples violate CVaR constraints.
└ @ Main.DistributionallyRobust /home/kanghyunryu/DRCC-MPC/src/distributionally_robust_controller.jl:315
┌ Warning: Time 3.70 [sec]: All samples violate CVaR constraints.
└ @ Main.DistributionallyRobust /home/kanghyunryu/DRCC-MPC/src/distributionally_robust_controller.jl:315
┌ Warning: Time 3.80 [sec]: All samples violate CVaR constraints.
└ @ Main.DistributionallyRobust /home/kanghyunryu/DRCC-MPC/src/distributionally_robust_controller.jl:315
┌ Warning: Time 3.90 [sec]: All samples violate CVaR constraints.
└ @ Main.DistributionallyRobust /home/kanghyunryu/DRCC-MPC/src/distributionally_robust_controller.jl:315
┌ Warning: Time 4.00 [sec]: All samples violate CVaR constraints.
└ @ Main.DistributionallyRobust /home/kanghyunryu/DRCC-MPC/src/distributionally_robust_controller.jl:315
┌ Warning: Time 4.10 [sec]: All samples violate CVaR constraints.
└ @ Main.DistributionallyRobust /home/kanghyunryu/DRCC-MPC/src/distributionally

Average computation time: 0.03877993106842041
std of computation time: 0.022430126825735294
List of computation time: Any[2.002716064453125e-5, 0.04281878471374512, 0.0314030647277832, 0.03441500663757324, 0.09769296646118164, 0.031774044036865234, 0.0340120792388916, 0.03507518768310547, 0.08585691452026367, 0.026618003845214844, 0.03253507614135742, 0.032708168029785156, 0.08627104759216309, 0.032531023025512695, 0.03432106971740723, 0.03307485580444336, 0.08353185653686523, 0.028820037841796875, 0.030377864837646484, 0.03126716613769531, 0.07868409156799316, 0.03233599662780762, 0.031466007232666016, 0.03199005126953125, 0.08751511573791504, 0.03105306625366211, 0.0318450927734375, 0.033776044845581055, 0.08337593078613281, 0.02926492691040039, 0.0315399169921875, 0.027599096298217773, 0.08573293685913086, 0.033560991287231445, 0.03140592575073242, 0.032421112060546875, 0.08605194091796875, 0.03176403045654297, 0.03327512741088867, 0.03249502182006836, 0.08683395385742188, 0.0325789

In [77]:
# display_log(result.log)

In [78]:
result.total_cnt_cost

1.194340999823127

In [79]:
result.total_pos_cost

185.1951049726166

In [80]:
result.total_col_cost

4.203139285490743

In [81]:
result.total_col

0

In [82]:
result.total_cnt_cost + result.total_pos_cost + result.total_col_cost

190.5925852579305

In [83]:
minimum([minimum(vcat([norm(get_position(w.e_state) - ap) for ap in values(w.ap_dict)], Inf))
                          for w in result.w_history])

0.8885722962972952

In [84]:
using CSV
for (idx, predict_dict) in enumerate(result.prediction_dict_history)
    if predict_dict != nothing
        CSV.write("./hotel_005/predict_dict_$idx.csv", predict_dict)
    end
end

position_history = [get_position(w.e_state) for w in result.w_history]
velocity_history = [get_velocity(w.e_state) for w in result.w_history]
open("./hotel_005/position_history.csv", "w") do file
    for row in position_history
        # Convert the row to a comma-separated string and write to file
        println(file, join(row, ","))
    end
end
open("./hotel_005/velocity_history.csv", "w") do file
    for row in velocity_history
        # Convert the row to a comma-separated string and write to file
        println(file, join(row, ","))
    end
end

In [85]:
# save("9_data_trajectron.jld2", "result", result)